# Synthetic reproducibility smoke test

This notebook verifies that the public modelling workflow can load data, apply
fold-specific preprocessing, fit the four model types, produce held-out
predictions, and calculate performance metrics without access to restricted
clinical records.

**Important:** Every record in the accompanying CSV is artificial. The smoke
test is not designed to reproduce or validate the manuscript estimates.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_SEED = 2026

# Support execution from either the repository root or notebooks directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "Data").exists() and (REPO_ROOT.parent / "Data").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_FILE = REPO_ROOT / "Data" / "Synthetic" / "synthetic_model_data.csv"
print(f"Repository root: {REPO_ROOT.resolve()}")
print(f"Synthetic data: {DATA_FILE.resolve()}")


In [ ]:
candidate_predictors = [
    "Age_at_Diagnosis", "Tumor_Size_mm", "Marital_Status", "Nationality",
    "Menopause_Status", "Breast_Cancer_Family_History", "Primary_Site_Group",
    "Laterality", "Histology_Group", "Tumor_Grade", "Summary_Stage",
    "ER_Status", "PR_Status", "HER2_Status", "Surgery", "Chemotherapy",
    "Hormone_Therapy",
]

core_predictors = [
    "Age_at_Diagnosis", "Tumor_Size_mm", "Tumor_Grade", "Summary_Stage",
    "ER_Status", "HER2_Status", "Surgery", "Chemotherapy", "Hormone_Therapy",
]

outcome_column = "Five_Year_Recurrence"
required_columns = set(candidate_predictors + [outcome_column])

data = pd.read_csv(DATA_FILE)
missing_columns = required_columns.difference(data.columns)
if missing_columns:
    raise ValueError(f"Synthetic dataset is missing: {sorted(missing_columns)}")

X = data[candidate_predictors].copy()
y = data[outcome_column].map({"No": 0, "Yes": 1})
if y.isna().any() or y.nunique() != 2:
    raise ValueError("Outcome must contain both No and Yes values.")

print(f"Synthetic records: {len(data):,}")
print(f"Synthetic events: {int(y.sum()):,}")
print(f"Synthetic event prevalence: {y.mean():.3f}")
data.head()


In [ ]:
numeric_predictors = ["Age_at_Diagnosis", "Tumor_Size_mm"]


def make_pipeline(selected_predictors, classifier):
    selected_numeric = [
        column for column in numeric_predictors if column in selected_predictors
    ]
    selected_categorical = [
        column for column in selected_predictors if column not in selected_numeric
    ]

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    preprocessing = ColumnTransformer([
        ("numeric", numeric_pipeline, selected_numeric),
        ("categorical", categorical_pipeline, selected_categorical),
    ])
    return Pipeline([
        ("preprocessing", preprocessing),
        ("classifier", classifier),
    ])


models = {
    "Core ridge logistic": (
        core_predictors,
        LogisticRegression(
            penalty="l2", C=1.0, solver="liblinear", max_iter=2000,
            random_state=RANDOM_SEED,
        ),
    ),
    "Elastic-net logistic": (
        candidate_predictors,
        LogisticRegression(
            penalty="elasticnet", l1_ratio=0.5, C=1.0, solver="saga",
            max_iter=3000, random_state=RANDOM_SEED,
        ),
    ),
    "Random forest": (
        candidate_predictors,
        RandomForestClassifier(
            n_estimators=200, min_samples_leaf=3, n_jobs=-1,
            random_state=RANDOM_SEED,
        ),
    ),
    "Gradient boosting": (
        candidate_predictors,
        GradientBoostingClassifier(
            n_estimators=100, learning_rate=0.05, max_depth=2,
            random_state=RANDOM_SEED,
        ),
    ),
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
print("Preprocessing pipelines and model configurations created.")


In [ ]:
results = []

for model_name, (selected_predictors, classifier) in models.items():
    pipeline = make_pipeline(selected_predictors, classifier)
    probabilities = cross_val_predict(
        pipeline,
        X[selected_predictors],
        y,
        cv=cv,
        method="predict_proba",
        n_jobs=-1,
    )[:, 1]

    results.append({
        "Model": model_name,
        "ROC_AUC": roc_auc_score(y, probabilities),
        "PR_AUC": average_precision_score(y, probabilities),
        "Brier_Score": brier_score_loss(y, probabilities),
        "Minimum_Probability": probabilities.min(),
        "Maximum_Probability": probabilities.max(),
    })

results_table = pd.DataFrame(results)
results_table.round(3)


In [ ]:
if len(results_table) != 4:
    raise AssertionError("All four model types were not evaluated.")
if not results_table[["ROC_AUC", "PR_AUC", "Brier_Score"]].notna().all().all():
    raise AssertionError("At least one smoke-test metric is missing.")

print("SMOKE TEST PASSED")
print("The public workflow executed on fully synthetic data.")
print("These values are not manuscript results and have no clinical interpretation.")
